In [13]:
import trimesh

mesh = trimesh.load("../data/test_house.obj", force="mesh")

print(type(mesh))
print("Граней:", len(mesh.faces))
print("Размеры (x, y, z):", mesh.extents)
print("Замкнутый ли меш:", mesh.is_watertight)

<class 'trimesh.base.Trimesh'>
Граней: 108
Размеры (x, y, z): [10.  15.   3.6]
Замкнутый ли меш: False


In [14]:
bbox = mesh.extents  # ширина, длина, высота охватывающей коробки

metrics = {
    "width": round(bbox[0], 2),
    "length": round(bbox[1], 2),
    "height": round(bbox[2], 2),
    "footprint_area": round(bbox[0] * bbox[1], 2),
    "bbox_volume": round(bbox[0] * bbox[1] * bbox[2], 2),
    "mesh_volume": round(mesh.volume, 2),
    "surface_area": round(mesh.area, 2),
    "height_to_width": round(bbox[2] / min(bbox[0], bbox[1]), 2),
}

for key, value in metrics.items():
    print(key, "=", value)

width = 10.0
length = 15.0
height = 3.6
footprint_area = 150.0
bbox_volume = 540.0
mesh_volume = 132.12
surface_area = 952.38
height_to_width = 0.36


In [15]:
normals = mesh.face_normals       # направление, куда смотрит каждая грань
areas = mesh.area_faces           # площадь каждой грани
centers = mesh.triangles_center   # центр каждой грани

def facade_info(axis, sign):
    # axis: 0 = x, 1 = y; sign: -1 = сторона к минимуму, +1 = к максимуму
    edge = mesh.bounds[0][axis] if sign < 0 else mesh.bounds[1][axis]
    looks_out = normals[:, axis] * sign > 0.9
    on_edge = abs(centers[:, axis] - edge) < 0.01
    actual = areas[looks_out & on_edge].sum()

    other = 0 if axis == 1 else 1
    full = mesh.extents[other] * mesh.extents[2]
    opening = full - actual
    return {
        "full_area": round(full, 2),
        "solid_area": round(actual, 2),
        "opening_area": round(opening, 2),
        "opening_ratio": round(opening / full, 2),
    }

facades = {
    "south": facade_info(1, -1),
    "north": facade_info(1, +1),
    "west": facade_info(0, -1),
    "east": facade_info(0, +1),
}

for side, info in facades.items():
    print(side, info)

metrics["facades"] = facades

south {'full_area': np.float64(36.0), 'solid_area': np.float64(30.0), 'opening_area': np.float64(6.0), 'opening_ratio': np.float64(0.17)}
north {'full_area': np.float64(36.0), 'solid_area': np.float64(36.0), 'opening_area': np.float64(0.0), 'opening_ratio': np.float64(0.0)}
west {'full_area': np.float64(54.0), 'solid_area': np.float64(54.0), 'opening_area': np.float64(0.0), 'opening_ratio': np.float64(0.0)}
east {'full_area': np.float64(54.0), 'solid_area': np.float64(54.0), 'opening_area': np.float64(0.0), 'opening_ratio': np.float64(0.0)}


In [17]:
import json

print(json.dumps(metrics, indent=2, ensure_ascii=False))
print("Предупреждения:")
for w_ in warnings:
    print(" -", w_)

{
  "width_m": 10.0,
  "length_m": 15.0,
  "height_m": 3.6,
  "footprint_area_m2": 150.0,
  "floors_estimate": 1,
  "floor_area_estimate_m2": 150.0,
  "mesh_volume_m3": 132.12,
  "height_to_width": 0.36,
  "facades": {
    "south": {
      "full_area": 36.0,
      "opening_area": 6.0,
      "opening_ratio": 0.17
    },
    "north": {
      "full_area": 36.0,
      "opening_area": 0.0,
      "opening_ratio": 0.0
    },
    "west": {
      "full_area": 54.0,
      "opening_area": 0.0,
      "opening_ratio": 0.0
    },
    "east": {
      "full_area": 54.0,
      "opening_area": 0.0,
      "opening_ratio": 0.0
    }
  },
  "watertight": false
}
Предупреждения:
 - Меш не замкнут: объём и площадь поверхности приблизительны
 - Этажность оценена по высоте, а не измерена
 - Площадь пола оценена по габаритам: для сложной формы плана она завышена


In [18]:
def facade_info(mesh, axis, sign):
    edge = mesh.bounds[0][axis] if sign < 0 else mesh.bounds[1][axis]
    looks_out = mesh.face_normals[:, axis] * sign > 0.9
    on_edge = abs(mesh.triangles_center[:, axis] - edge) < 0.01
    actual = mesh.area_faces[looks_out & on_edge].sum()

    other = 0 if axis == 1 else 1
    full = mesh.extents[other] * mesh.extents[2]
    opening = full - actual
    return {
        "full_area": round(float(full), 2),
        "opening_area": round(float(opening), 2),
        "opening_ratio": round(float(opening / full), 2),
    }


def analyze_mesh(path, floor_height=3.0):
    mesh = trimesh.load(path, force="mesh")
    w, l, h = (float(v) for v in mesh.extents)
    warnings = []

    if not mesh.is_watertight:
        warnings.append("Меш не замкнут: объём и площадь поверхности приблизительны")
    if max(w, l, h) > 200:
        warnings.append("Размеры очень большие: возможно, модель в миллиметрах, а не в метрах")
    if h > max(w, l) * 3:
        warnings.append("Модель сильно вытянута по высоте: возможно, вертикальная ось не та (Y вместо Z)")

    floors = max(1, round(h / floor_height))
    footprint = w * l
    warnings.append("Этажность оценена по высоте, а не измерена")
    warnings.append("Площадь пола оценена по габаритам: для сложной формы плана она завышена")

    metrics = {
        "width_m": round(w, 2),
        "length_m": round(l, 2),
        "height_m": round(h, 2),
        "footprint_area_m2": round(footprint, 2),
        "floors_estimate": floors,
        "floor_area_estimate_m2": round(footprint * floors, 2),
        "mesh_volume_m3": round(float(mesh.volume), 2),
        "height_to_width": round(h / min(w, l), 2),
        "facades": {
            "south": facade_info(mesh, 1, -1),
            "north": facade_info(mesh, 1, +1),
            "west": facade_info(mesh, 0, -1),
            "east": facade_info(mesh, 0, +1),
        },
        "watertight": bool(mesh.is_watertight),
    }
    return metrics, warnings

In [19]:
for ext in ["obj", "stl", "glb"]:
    m, w_ = analyze_mesh(f"../data/test_house.{ext}")
    print(ext, m["width_m"], m["length_m"], m["height_m"], m["facades"]["south"]["opening_area"])

obj 10.0 15.0 3.6 6.0
stl 10.0 15.0 3.6 6.0
glb 10.0 15.0 3.6 6.0
